# 01 - Data exploration (HAM10000)

Goals of this notebook:

1. Look at the class distribution (imbalance).
2. See how many images belong to the same lesion (why we split by `lesion_id`).
3. Check the lesion-level train/val/test split.
4. Look at example images per class.

Download the data first: `python scripts/download_data.py` (see `data/README.md`).

**Research/learning project - not a medical device.**

In [ ]:
import os
import sys
from pathlib import Path

# Run from the repository root so that relative data paths and `src` imports work.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
from PIL import Image

from src.dataset import HAM_CLASSES, MALIGNANT, load_metadata, split_by_lesion
from src.utils import load_config

cfg = load_config()  # use load_config("configs/smoke.yaml") to try the synthetic data
df = load_metadata(cfg["data"]["metadata_csv"], cfg["data"]["root"], mode="multiclass")
print(f"{len(df)} images, {df['lesion_id'].nunique()} lesions")
df.head()

## Class distribution

The dataset is strongly imbalanced. This is why accuracy alone is misleading and why training uses class weights or a weighted sampler.

In [ ]:
counts = df["dx"].value_counts().reindex(HAM_CLASSES)
colors = ["tab:red" if c in MALIGNANT else "tab:blue" for c in counts.index]

ax = counts.plot.bar(color=colors, figsize=(7, 3.5))
ax.set_title("Images per diagnosis (red = counted as malignant in binary mode)")
ax.set_ylabel("images")
plt.show()

print((counts / counts.sum()).round(3).to_string())
print(f"\nMalignant share (binary mode): {df['dx'].isin(MALIGNANT).mean():.3f}")

## Images per lesion

Many lesions were photographed several times. If images of one lesion ended up in both the training and the test set, the test score would be too optimistic (data leakage).

In [ ]:
per_lesion = df.groupby("lesion_id").size()
print(per_lesion.value_counts().sort_index().rename("lesions").rename_axis("images per lesion").to_string())
print(f"\nImages that share their lesion with another image: {(per_lesion[per_lesion > 1].sum() / len(df)):.1%}")

# Does every lesion have exactly one diagnosis? (required for the stratified split)
assert (df.groupby("lesion_id")["dx"].nunique() == 1).all()

## Lesion-level split

In [ ]:
import pandas as pd

splits = split_by_lesion(df, cfg["data"]["val_size"], cfg["data"]["test_size"], cfg["seed"])

table = pd.DataFrame({name: part["dx"].value_counts() for name, part in splits.items()}).reindex(HAM_CLASSES)
display(table)
display((table / table.sum()).round(3))  # class shares should be similar in all splits

lesions = {name: set(part["lesion_id"]) for name, part in splits.items()}
assert not (lesions["train"] & lesions["val"] or lesions["train"] & lesions["test"] or lesions["val"] & lesions["test"])
print("No lesion appears in more than one split.")

## Metadata: age, sex, body site, type of ground truth

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
df["age"].plot.hist(bins=18, ax=axes[0], title="Age")
df["sex"].value_counts().plot.bar(ax=axes[1], title="Sex")
df["localization"].value_counts().plot.bar(ax=axes[2], title="Body site")
plt.tight_layout()
plt.show()

# How was the diagnosis established? (histo = histopathology)
pd.crosstab(df["dx"], df["dx_type"])

## Example images per class

In [ ]:
n = 5
fig, axes = plt.subplots(len(HAM_CLASSES), n, figsize=(2.2 * n, 2 * len(HAM_CLASSES)))
for row, dx in enumerate(HAM_CLASSES):
    subset = df[df["dx"] == dx]
    sample = subset.sample(n=min(n, len(subset)), random_state=cfg["seed"])
    for col in range(n):
        ax = axes[row, col]
        ax.axis("off")
        if col < len(sample):
            ax.imshow(Image.open(sample.iloc[col]["path"]))
        if col == 0:
            ax.set_title(dx, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## Things to keep in mind

- The dataset contains almost only light skin and has no skin-type labels.
- All images are dermatoscopic; phone photos look very different.
- Look for possible shortcuts in the images above (rulers, hairs, dark corners, ink marks).